# Sesión 06 — Pushover universal

Este notebook resuelve gravedad, calibración axial y pushover para el caso definido en `MI_EDIFICIO`.


In [1]:
from pathlib import Path
import sys, json, numpy as np
RAIZ=Path.cwd()
while not ((RAIZ/'temario.md').is_file() and (RAIZ/'PUSHOVER_PORTABLE').is_dir()): RAIZ=RAIZ.parent
PORTABLE=RAIZ/'PUSHOVER_PORTABLE'; sys.path.insert(0,str(PORTABLE))
from ejecutar_pushover import cargar_entrada_maestra, evaluar_calidad_curva
from sesion_03.biblioteca_rotulas import leer_contrato_sesion03
from sesion_05.contrato_portable import guardar_contrato_sesion05
from sesion_06.caso_medrano import crear_caso_gravitacional_desde_datos
from sesion_06.pushover import ConfiguracionPushover, analizar_pushover, crear_patron_lateral, guardar_resultados
from sesion_06.rotulas_axiales import calibrar_contrato_nivel1
from sesion_06.criterios_externos import aplicar_criterios_externos
from sesion_06.visualizacion import guardar_figuras
RUTA_MAESTRA=PORTABLE/'casos'/'edificio_6pisos'/'entrada_maestra.json'
entrada,_,cargas=cargar_entrada_maestra(RUTA_MAESTRA); CASO_ID=entrada['caso_id']
base=RAIZ/'resultados_curso_portable'/CASO_ID
biblioteca=leer_contrato_sesion03(base/'sesion_03'/'contrato_sesion03.json')['biblioteca_rotulas']
contrato=json.loads((base/'sesion_05'/'contrato_sesion05.json').read_text(encoding='utf-8'))
caso_gravedad=crear_caso_gravitacional_desde_datos(contrato,cargas,entrada)
cfg=ConfiguracionPushover(**entrada['pushover'].get('configuracion',{}))
asignacion={int(k):str(v) for k,v in contrato['asignacion_secciones'].items()}
ccfg=entrada.get('calibracion',{})
cal=calibrar_contrato_nivel1(contrato,caso_gravedad,asignacion,biblioteca,configuracion=cfg,fuera_de_rango=str(ccfg.get('fuera_de_rango','error')),actualizar_EI=bool(ccfg.get('actualizar_EI',True)),tolerancia_axial_fraccion=float(ccfg.get('tolerancia_axial_fraccion',0.01)),max_iteraciones=int(ccfg.get('max_iteraciones',20)),catalogo_secciones=entrada['secciones'])
if not cal['convergio']: raise RuntimeError('La calibración axial no convergió.')
contrato_cal=aplicar_criterios_externos(cal['contrato'],entrada['_catalogo_criterios'],permitir_provisionales=bool(entrada.get('configuracion_criterios',{}).get('permitir_provisionales',False)),politica_exceso_capacidad=str(entrada.get('configuracion_criterios',{}).get('politica_exceso_capacidad','error')))
n=int(contrato_cal['geometria']['n_pisos']); definicion=entrada['pushover'].get('patron_lateral','triangular')
coef=np.asarray(definicion,dtype=float) if isinstance(definicion,list) else (np.arange(1,n+1,dtype=float) if definicion=='triangular' else np.ones(n))
patron=crear_patron_lateral(contrato_cal,coef)
resultados=analizar_pushover(contrato_cal,patron,cargas_gravitacionales=caso_gravedad,configuracion=cfg)
resultados['calidad_curva']=evaluar_calidad_curva(resultados,entrada['pushover'].get('calidad_curva',{}))
salida=base/'sesion_06'; salida.mkdir(parents=True,exist_ok=True)
guardar_contrato_sesion05(contrato_cal,base/'sesion_05'/'contrato_sesion05_calibrado.json')
guardar_resultados(resultados,salida/'resultados_pushover.json'); guardar_figuras(contrato_cal,resultados,salida)
hist=resultados['historia']; print('Caso:',CASO_ID,'| convergió:',resultados['convergio'],'| parada:',resultados['razon_parada'],'| puntos:',len(hist))
print('Resultados:',salida)


Caso: edificio_6pisos | convergió: True | parada: desplazamiento_maximo | puntos: 141
Resultados: d:\DOCENCIA\PRINBEL\PUSHOVER PORTICOS\resultados_curso_portable\edificio_6pisos\sesion_06
